# 🛰️ Siamese-SNN Change Detection — V2 FIXED

**V2 Fixes:**
- CosineAnnealingLR (LR won't die)
- Diagnostic output monitoring
- Adaptive threshold
- Proper warm-up

**Runtime → Change runtime type → T4 GPU** before running!

In [ ]:
!pip install -q torch torchvision snntorch rasterio pillow scipy tqdm

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Runtime → Change runtime type → T4 GPU")

In [ ]:
# Upload dataset
from google.colab import files
import os
from pathlib import Path

print("Upload your oscd_dataset.zip:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/data/oscd
!unzip -q "{zip_name}" -d /content/data/oscd/

# Auto-detect nested folders
oscd_root = Path("/content/data/oscd")
cities_expected = ["aguasclaras", "bercy", "bordeaux", "paris", "mumbai"]
if not any((oscd_root / c).exists() for c in cities_expected):
    for sub in oscd_root.iterdir():
        if sub.is_dir() and any((sub / c).exists() for c in cities_expected):
            oscd_root = sub
            break

found = [d.name for d in oscd_root.iterdir() if d.is_dir()]
print(f"\n✅ Found {len(found)} city folders at: {oscd_root}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional
import numpy as np

try:
    import snntorch as snn
    from snntorch import surrogate, spikegen
    import snntorch.functional as SF
    HAS_SNNTORCH = True
    print(f'snntorch version: {snn.__version__}')
except ImportError:
    HAS_SNNTORCH = False
    print('snntorch not found — using ReLU fallback')

# ============================================================
#  ENCODER
# ============================================================
class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
        )
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        f = self.conv(x)
        return f, self.pool(f)

class SiameseEncoder(nn.Module):
    def __init__(self, in_channels=4, encoder_channels=None):
        super().__init__()
        if encoder_channels is None: encoder_channels = [64,128,256,512]
        self.encoder_channels = encoder_channels
        chs = [in_channels] + encoder_channels
        self.blocks = nn.ModuleList([EncoderBlock(chs[i], chs[i+1]) for i in range(len(encoder_channels))])
        self.bottleneck = nn.Sequential(
            nn.Conv2d(encoder_channels[-1], encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2), nn.ReLU(True),
            nn.Conv2d(encoder_channels[-1]*2, encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2), nn.ReLU(True),
        )
    def encode_single(self, x):
        skips = []
        for b in self.blocks:
            f, x = b(x)
            skips.append(f)
        return self.bottleneck(x), skips
    def forward(self, a, b):
        ba, sa = self.encode_single(a)
        bb, sb = self.encode_single(b)
        return torch.abs(ba - bb), [torch.abs(x-y) for x,y in zip(sa, sb)]

# ============================================================
#  SNN DECODER
# ============================================================
class SNNDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, beta=0.9):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv1 = nn.Sequential(nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        self.conv2 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        if HAS_SNNTORCH:
            sg = surrogate.fast_sigmoid(slope=25)
            self.lif1 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=False)
            self.lif2 = snn.Leaky(beta=beta, spike_grad=sg, init_hidden=False)
        else:
            self.lif1 = self.lif2 = None
            self.relu = nn.ReLU(True)
    def forward(self, x, skip, m1=None, m2=None):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        if HAS_SNNTORCH and self.lif1:
            if m1 is None: m1 = self.lif1.init_leaky()
            s1, m1 = self.lif1(x, m1)
            x = self.conv2(s1)
            if m2 is None: m2 = self.lif2.init_leaky()
            s2, m2 = self.lif2(x, m2)
            return s2, m1, m2
        else:
            x = self.relu(x); x = self.conv2(x); x = self.relu(x)
            return x, None, None

class SNNDecoder(nn.Module):
    def __init__(self, enc_chs=None, num_classes=2, beta=0.9, num_steps=20):
        super().__init__()
        if enc_chs is None: enc_chs = [64,128,256,512]
        self.num_steps = num_steps
        bot_ch = enc_chs[-1]*2
        self.blocks = nn.ModuleList()
        in_ch = bot_ch
        for i in range(len(enc_chs)-1, -1, -1):
            self.blocks.append(SNNDecoderBlock(in_ch, enc_chs[i], enc_chs[i], beta))
            in_ch = enc_chs[i]
        self.head = nn.Conv2d(enc_chs[0], num_classes, 1)
    def forward(self, spk_in, skips):
        T = spk_in.shape[0] if spk_in.dim()==5 else self.num_steps
        rev_skips = list(reversed(skips))
        spk_rec, mem_rec = [], []
        block_mems = [(None,None) for _ in self.blocks]
        for t in range(T):
            x = spk_in[t] if spk_in.dim()==5 else spk_in
            for i,(blk,sk) in enumerate(zip(self.blocks, rev_skips)):
                m1,m2 = block_mems[i]
                x,m1,m2 = blk(x, sk, m1, m2)
                block_mems[i] = (m1,m2)
            out = self.head(x)
            spk_rec.append(out)
            mem_rec.append(out.clone())
        return spk_rec, mem_rec

# ============================================================
#  FULL MODEL
# ============================================================
class SiameseSNN(nn.Module):
    def __init__(self, in_channels=4, encoder_channels=None, num_classes=2, beta=0.9, num_steps=20):
        super().__init__()
        if encoder_channels is None: encoder_channels = [64,128,256,512]
        self.num_steps = num_steps
        self.encoder = SiameseEncoder(in_channels, encoder_channels)
        self.decoder = SNNDecoder(encoder_channels, num_classes, beta, num_steps)

    def forward(self, a, b):
        diff_bot, diff_skips = self.encoder(a, b)
        if HAS_SNNTORCH:
            diff_norm = torch.sigmoid(diff_bot)
            spike_trains = spikegen.rate(diff_norm, num_steps=self.num_steps)
        else:
            spike_trains = diff_bot.unsqueeze(0).repeat(self.num_steps,1,1,1,1)
        return self.decoder(spike_trains, diff_skips)

    def predict(self, a, b, threshold=0.3):
        self.eval()
        with torch.no_grad():
            spk_rec, _ = self.forward(a, b)
            spk_stack = torch.stack(spk_rec, dim=0)
            firing_rate = spk_stack.mean(dim=0)  # (B, 2, H, W)
            # Use argmax instead of threshold — more robust
            pred = firing_rate.argmax(dim=1)  # (B, H, W) — 0 or 1
        return pred

    def get_change_prob(self, a, b):
        """Get raw change probability map for diagnostics."""
        self.eval()
        with torch.no_grad():
            spk_rec, _ = self.forward(a, b)
            spk_stack = torch.stack(spk_rec, dim=0)
            firing_rate = spk_stack.mean(dim=0)
            probs = torch.softmax(firing_rate, dim=1)
            return probs[:, 1]  # change probability

print('✅ Model loaded')

In [ ]:
# ============================================================
#  LOSS — Using standard CrossEntropy (most reliable)
# ============================================================

class ChangeDetectionLoss(nn.Module):
    """Simple, reliable loss for imbalanced change detection.
    Uses standard CrossEntropyLoss with class weights + Dice.
    No autocast issues, no BCE complications."""

    def __init__(self, change_weight=15.0, dice_weight=0.3):
        super().__init__()
        self.dice_weight = dice_weight
        # Class weights: [no_change=1.0, change=15.0]
        self.register_buffer('class_weights', torch.tensor([1.0, change_weight]))

    def forward(self, spk_recordings, targets):
        # Average firing rate over time steps
        spk_stack = torch.stack(spk_recordings, dim=0)  # (T, B, 2, H, W)
        logits = spk_stack.mean(dim=0)  # (B, 2, H, W)

        # Standard weighted cross-entropy (autocast-safe)
        B, C, H, W = logits.shape
        ce_loss = F.cross_entropy(
            logits, targets.long(),
            weight=self.class_weights.to(logits.device)
        )

        # Dice loss for spatial coherence
        probs = torch.softmax(logits, dim=1)
        change_prob = probs[:, 1]
        targets_f = targets.float()
        intersection = (change_prob * targets_f).sum()
        union = change_prob.sum() + targets_f.sum()
        dice_loss = 1.0 - (2.0 * intersection + 1.0) / (union + 1.0)

        return ce_loss + self.dice_weight * dice_loss

print('✅ Loss function loaded (autocast-safe, weighted CE + Dice)')

In [ ]:
# ============================================================
#  DATASET — Same as before
# ============================================================
from torch.utils.data import Dataset
from pathlib import Path

try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False

try:
    from scipy.ndimage import zoom as scipy_zoom
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

TRAIN_CITIES = ['aguasclaras','bercy','bordeaux','nantes','paris','rennes',
                'saclay_e','abudhabi','cupertino','pisa','beihai','hongkong','beirut','mumbai']
TEST_CITIES = ['brasilia','montpellier','norcia','rio','saclay_w',
               'valencia','dubai','lasvegas','milano','chongqing']

def _find_band(d, bn):
    d = Path(d)
    p = d / f'{bn}.tif'
    if p.exists(): return p
    m = list(d.glob(f'*_{bn}.tif'))
    if m: return m[0]
    m = list(d.glob(f'*_{bn.upper()}.tif'))
    if m: return m[0]
    return None

class OSCDDataset(Dataset):
    def __init__(self, root, split='train', patch_size=128, bands=None, augment=True):
        self.root = Path(root)
        self.split = split
        self.ps = patch_size
        self.bands = bands or ['B02','B03','B04','B08']
        self.augment = augment and split=='train'
        self._dims = {}
        if split=='train':
            self.cities = [c for c in TRAIN_CITIES if (self.root/c).exists()]
        else:
            self.cities = [c for c in TEST_CITIES if (self.root/c).exists()]
        self.use_mock = len(self.cities)==0
        self.patches = self._index()
        print(f'{split}: {len(self.cities)} cities, {len(self.patches)} patches')

    def _refdim(self, city):
        if city in self._dims: return self._dims[city]
        h,w = 512,512
        for b in ['B02','B03','B04','B08']:
            f = _find_band(self.root/city/'imgs_1', b)
            if f and HAS_RASTERIO:
                with rasterio.open(str(f)) as s: h,w = s.height, s.width
                break
        self._dims[city] = (h,w)
        return h,w

    def _index(self):
        if self.use_mock:
            return [{'city':'mock','row':0,'col':0}]*100
        patches = []
        for c in self.cities:
            h,w = self._refdim(c)
            stride = self.ps//2 if self.split=='train' else self.ps
            for r in range(0, h-self.ps+1, stride):
                for co in range(0, w-self.ps+1, stride):
                    patches.append({'city':c,'row':r,'col':co})
        return patches

    def _loadbands(self, city, td):
        d = self.root/city/td
        rh,rw = self._refdim(city)
        arrs = []
        for bn in self.bands:
            f = _find_band(d, bn)
            if f and HAS_RASTERIO:
                with rasterio.open(str(f)) as s: data = s.read(1).astype(np.float32)
                if data.shape != (rh,rw):
                    if HAS_SCIPY:
                        data = scipy_zoom(data, (rh/data.shape[0], rw/data.shape[1]), order=1)[:rh,:rw]
                        if data.shape[0]<rh or data.shape[1]<rw:
                            data = np.pad(data, ((0,rh-data.shape[0]),(0,rw-data.shape[1])))
                arrs.append(data)
            else:
                arrs.append(np.zeros((rh,rw), dtype=np.float32))
        return np.stack(arrs, 0)

    def _loadmask(self, city):
        cm = self.root/city/'cm'/'cm.png'
        rh,rw = self._refdim(city)
        if cm.exists():
            from PIL import Image
            m = np.array(Image.open(str(cm)))
            if m.ndim==3: m = m[:,:,0]
            m = (m>128).astype(np.float32)
            if m.shape != (rh,rw):
                from PIL import Image as PI
                mp = PI.fromarray((m*255).astype(np.uint8)).resize((rw,rh), PI.NEAREST)
                m = (np.array(mp)>128).astype(np.float32)
        else:
            m = np.zeros((rh,rw), dtype=np.float32)
        return m

    def __len__(self): return len(self.patches)

    def __getitem__(self, idx):
        p = self.patches[idx]
        if self.use_mock:
            C,H = len(self.bands), self.ps
            return torch.rand(C,H,H), torch.rand(C,H,H), torch.randint(0,2,(H,H))
        city,r,c = p['city'],p['row'],p['col']
        a = self._loadbands(city,'imgs_1')
        b = self._loadbands(city,'imgs_2')
        m = self._loadmask(city)
        a = a[:, r:r+self.ps, c:c+self.ps]
        b = b[:, r:r+self.ps, c:c+self.ps]
        m = m[r:r+self.ps, c:c+self.ps]
        _,ph,pw = a.shape
        if ph<self.ps or pw<self.ps:
            a = np.pad(a,((0,0),(0,self.ps-ph),(0,self.ps-pw)))
            b = np.pad(b,((0,0),(0,self.ps-ph),(0,self.ps-pw)))
            m = np.pad(m,((0,self.ps-ph),(0,self.ps-pw)))
        # Normalize per band
        for ch in range(a.shape[0]):
            for img in [a,b]:
                mn,mx = img[ch].min(), img[ch].max()
                if mx>mn: img[ch] = (img[ch]-mn)/(mx-mn)
        if self.augment:
            if np.random.random()>0.5: a=np.flip(a,2).copy(); b=np.flip(b,2).copy(); m=np.flip(m,1).copy()
            if np.random.random()>0.5: a=np.flip(a,1).copy(); b=np.flip(b,1).copy(); m=np.flip(m,0).copy()
            k=np.random.randint(0,4)
            if k>0: a=np.rot90(a,k,(1,2)).copy(); b=np.rot90(b,k,(1,2)).copy(); m=np.rot90(m,k,(0,1)).copy()
        return torch.from_numpy(a.copy()).float(), torch.from_numpy(b.copy()).float(), torch.from_numpy(m.copy()).long()

print('✅ Dataset ready')

In [ ]:
# ============================================================
#  CONFIG
# ============================================================

GPU_CONFIG = {
    'epochs': 60,
    'batch_size': 8,
    'lr': 1e-3,              # Higher LR — let it learn!
    'weight_decay': 1e-4,
    'patch_size': 128,
    'num_steps': 10,         # 10 steps (20 was too many — sparse spikes)
    'num_bands': 4,
    'encoder_channels': [32, 64, 128, 256],  # Smaller = faster convergence
    'gradient_clip': 1.0,
    'change_weight': 15.0,
}

USE_BANDS = ['B02', 'B03', 'B04', 'B08']
print('Config:', GPU_CONFIG)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  TRAINING — FIXED VERSION
# ══════════════════════════════════════════════════════════════

import time, os
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Dataset
train_ds = OSCDDataset(str(oscd_root), 'train', GPU_CONFIG['patch_size'], USE_BANDS, True)
val_ds = OSCDDataset(str(oscd_root), 'test', GPU_CONFIG['patch_size'], USE_BANDS, False)

# If no labeled test cities, split train
if len(val_ds)==0 or val_ds.use_mock:
    print('Splitting train 80/20 for validation')
    n = len(train_ds); t = int(0.8*n)
    train_ds, val_ds = torch.utils.data.random_split(train_ds, [t, n-t])

train_loader = DataLoader(train_ds, batch_size=GPU_CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=GPU_CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} patches | Val: {len(val_ds)} patches')

# Model
model = SiameseSNN(
    in_channels=GPU_CONFIG['num_bands'],
    encoder_channels=GPU_CONFIG['encoder_channels'],
    beta=0.9,
    num_steps=GPU_CONFIG['num_steps'],
).to(device)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')

# Loss / Optimizer / Scheduler
loss_fn = ChangeDetectionLoss(change_weight=GPU_CONFIG['change_weight'])
loss_fn = loss_fn.to(device)
optimizer = AdamW(model.parameters(), lr=GPU_CONFIG['lr'], weight_decay=GPU_CONFIG['weight_decay'])
# CosineAnnealingWarmRestarts: LR stays active, restarts every T_0 epochs
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-5)
scaler = GradScaler('cuda')

os.makedirs('/content/outputs/models', exist_ok=True)
best_f1 = 0.0
history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_iou':[]}

print('='*70)
print(f'  🚀 Training: {GPU_CONFIG["epochs"]} epochs | LR={GPU_CONFIG["lr"]} | T={GPU_CONFIG["num_steps"]}')
print('='*70)

for epoch in range(1, GPU_CONFIG['epochs']+1):
    t0 = time.time()

    # ── Train ──
    model.train()
    total_loss = 0.0
    for bi, batch in enumerate(train_loader):
        img_a = batch[0].to(device)
        img_b = batch[1].to(device)
        mask  = batch[2].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            spk_rec, _ = model(img_a, img_b)
            loss = loss_fn(spk_rec, mask)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GPU_CONFIG['gradient_clip'])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()
    avg_train = total_loss / len(train_loader)
    history['train_loss'].append(avg_train)

    # ── Validate ──
    model.eval()
    tp=fp=fn=tn=0
    vl_total=0.0
    with torch.no_grad():
        for batch in val_loader:
            ia = batch[0].to(device).float()
            ib = batch[1].to(device).float()
            mk = batch[2].to(device).long()
            with torch.amp.autocast('cuda'):
                sr, _ = model(ia, ib)
                vl_total += loss_fn(sr, mk).item()
            # Use argmax prediction (no threshold needed)
            pred = model.predict(ia, ib)
            tp += ((pred==1)&(mk==1)).sum().item()
            fp += ((pred==1)&(mk==0)).sum().item()
            fn += ((pred==0)&(mk==1)).sum().item()
            tn += ((pred==0)&(mk==0)).sum().item()

    prec = tp/(tp+fp+1e-8)
    rec  = tp/(tp+fn+1e-8)
    f1   = 2*prec*rec/(prec+rec+1e-8)
    iou  = tp/(tp+fp+fn+1e-8)
    avg_val = vl_total/len(val_loader)
    lr_now = optimizer.param_groups[0]['lr']

    history['val_loss'].append(avg_val)
    history['val_f1'].append(f1)
    history['val_iou'].append(iou)

    elapsed = time.time()-t0
    print(f'Ep {epoch:3d}/{GPU_CONFIG["epochs"]} | '
          f'TrL: {avg_train:.4f} | VL: {avg_val:.4f} | '
          f'F1: {f1:.4f} IoU: {iou:.4f} | P:{prec:.3f} R:{rec:.3f} | '
          f'LR: {lr_now:.2e} | {elapsed:.0f}s')

    # ── Diagnostics every 5 epochs ──
    if epoch % 5 == 1 or f1 > 0:
        with torch.no_grad():
            sample_a = batch[0][:1].to(device)
            sample_b = batch[1][:1].to(device)
            cp = model.get_change_prob(sample_a, sample_b)
            pred_pix = model.predict(sample_a, sample_b).sum().item()
            print(f'    📊 change_prob: min={cp.min():.4f} max={cp.max():.4f} mean={cp.mean():.4f} | pred_change_pixels={pred_pix}')

    # Save best
    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_f1': best_f1,
            'config': GPU_CONFIG,
        }, '/content/outputs/models/best_model.pt')
        print(f'    ★ New best! F1={best_f1:.4f}')

    if epoch % 10 == 0:
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'best_f1':best_f1,'config':GPU_CONFIG},
                   f'/content/outputs/models/checkpoint_epoch_{epoch}.pt')

print(f'\n{"="*70}')
print(f'✅ Done! Best F1: {best_f1:.4f}')
print(f'{"="*70}')

In [ ]:
# Training curves
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3,figsize=(18,5))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_f1'], color='green', linewidth=2)
axes[1].set_title(f'F1 (best={best_f1:.4f})'); axes[1].grid(alpha=0.3)
axes[2].plot(history['val_iou'], color='purple', linewidth=2)
axes[2].set_title('IoU'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/content/outputs/training_curves.png', dpi=150); plt.show()

In [ ]:
# Download model
from google.colab import files
files.download('/content/outputs/models/best_model.pt')
print('📥 Place in: outputs/models/best_model.pt')